# 🍜 NHAM EY KOR BAN — Phnom Penh Food Recommendation System
### Final Project Notebook — Data Structure + Algorithm + Visualisation

---
> **"Nham Ey Kor Ban?"** (ញុាំអីក៏បាន) means *"I can eat anything!"* in Khmer.  
> This notebook documents the full project: dataset design, filtering algorithm, scoring logic, and a static visualisation of results — running without any GUI.

**Tech Stack:** Python · Matplotlib · NumPy  
**Dataset:** 100+ hand-curated Phnom Penh food entries  
**Location:** Phnom Penh, Cambodia 🇰🇭

---
## 1. Dependencies & Imports
Install required packages (only needed once). The main GUI app requires `tkintermapview` and `Pillow`, but this notebook uses only `matplotlib` and `numpy` for the static visualisation.

In [ ]:
# Install dependencies (run once)
# %pip install matplotlib numpy pillow tkintermapview

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from IPython.display import display, HTML

# Inline charts in the notebook
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.family'] = 'sans-serif'

print('✅ Imports complete')

---
## 2. Dataset — `FOOD_DATASET`

### 2.1 Data Structure Design

The dataset uses a **List of Dictionaries** — the primary data structure of this project.

| Key | Type | Description |
|-----|------|-------------|
| `name` | `str` | Dish / restaurant name |
| `place` | `str` | Venue in Phnom Penh |
| `area` | `str` | City district (BKK1, Daun Penh, etc.) |
| `lat` / `lng` | `float` | GPS coordinates for map |
| `budget` | `str` | `'low'` / `'mid'` / `'high'` |
| `budget_min/max` | `int` | Price range in USD |
| `taste` | `str` | `savory`, `spicy`, `sweet`, `sour`, `mild` |
| `main` | `str` | Main category: `rice`, `noodles`, `soup`, `beef` … |
| `cuisine` | `str` | `khmer`, `chinese`, `western` … |
| `allergens` | `list[str]` | e.g. `['fish', 'dairy']` |
| `dietary` | `list[str]` | `vegan`, `vegetarian` |
| `emoji` | `str` | UI visual shorthand |
| `desc` | `str` | Short description |
| `price` | `str` | Formatted price string |

**Why a list of dicts?**  
- O(n) linear iteration for filtering  
- O(1) key access per item for scoring  
- Human-readable and easily extendable

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  CONSTANTS
# ─────────────────────────────────────────────────────────────────────
CUISINES = [
    "any", "khmer", "chinese", "vietnamese",
    "thai", "indian", "western", "italian",
    "japanese", "korean", "french", "international",
    "seafood"
]

ALLERGIES = [
    "gluten", "dairy", "egg", "peanuts",
    "soy", "fish", "shellfish", "pork"
]

# ─────────────────────────────────────────────────────────────────────
#  FOOD DATASET  (abridged inline — full dataset in dataSet.py)
# ─────────────────────────────────────────────────────────────────────
FOOD_DATASET = [
    # ── KHMER LOCAL ──────────────────────────────────────────────────
    {"name": "Nom Banh Chok",          "place": "Kandal Market",              "area": "Daun Penh",   "lat": 11.5668, "lng": 104.9250, "budget": "low",  "budget_min": 1,  "budget_max": 2,  "taste": "savory", "main": "noodles",  "cuisine": "khmer",         "allergens": ["fish"],                  "dietary": [],                     "emoji": "🍜", "desc": "Fresh rice noodles with green fish curry & herbs",          "price": "$1–$2"},
    {"name": "Bai Sach Chrouk",        "place": "Orussey Market",            "area": "Chamkar Mon", "lat": 11.5616, "lng": 104.9191, "budget": "low",  "budget_min": 1,  "budget_max": 3,  "taste": "savory", "main": "rice",     "cuisine": "khmer",         "allergens": ["soy"],                   "dietary": [],                     "emoji": "🍚", "desc": "Grilled pork on steamed rice with pickled veggies",           "price": "$1.50–$2.50"},
    {"name": "Kuy Teav",               "place": "Sorya Food Court",          "area": "BKK1",        "lat": 11.5547, "lng": 104.9242, "budget": "low",  "budget_min": 1,  "budget_max": 3,  "taste": "savory", "main": "noodles",  "cuisine": "khmer",         "allergens": ["pork", "shellfish"],     "dietary": [],                     "emoji": "🍲", "desc": "Cambodian pork noodle soup with bean sprouts & herbs",         "price": "$1.50–$3"},
    {"name": "Amok Trey",              "place": "Romdeng Restaurant",        "area": "BKK1",        "lat": 11.5588, "lng": 104.9240, "budget": "mid",  "budget_min": 6,  "budget_max": 9,  "taste": "savory", "main": "seafood",  "cuisine": "khmer",         "allergens": ["fish", "egg", "dairy"], "dietary": [],                     "emoji": "🐟", "desc": "Steamed coconut fish curry in banana leaf",                   "price": "$6–$9"},
    {"name": "Lok Lak",                "place": "Dara Reang Sey",            "area": "Daun Penh",   "lat": 11.5625, "lng": 104.9301, "budget": "mid",  "budget_min": 5,  "budget_max": 8,  "taste": "savory", "main": "beef",     "cuisine": "khmer",         "allergens": ["soy", "egg"],            "dietary": [],                     "emoji": "🥩", "desc": "Stir-fried beef cubes with lime-pepper sauce & fried egg",    "price": "$5–$8"},
    {"name": "Bobor Khmer",            "place": "BKK3 Market",              "area": "BKK1",        "lat": 11.5547, "lng": 104.9242, "budget": "low",  "budget_min": 1,  "budget_max": 2,  "taste": "mild",   "main": "soup",     "cuisine": "khmer",         "allergens": ["fish"],                  "dietary": [],                     "emoji": "🥣", "desc": "Cambodian rice porridge with ginger, fish & garnishes",       "price": "$1–$2"},
    {"name": "Grilled Corn",           "place": "Riverside Stalls",         "area": "Daun Penh",   "lat": 11.5700, "lng": 104.9285, "budget": "low",  "budget_min": 1,  "budget_max": 1,  "taste": "sweet",  "main": "snack",    "cuisine": "khmer",         "allergens": [],                        "dietary": ["vegan","vegetarian"], "emoji": "🌽", "desc": "Charcoal-grilled corn rubbed with coconut milk & salt",        "price": "$0.50–$1"},
    {"name": "Khmer BBQ",              "place": "BBQ Street 278",           "area": "Chamkar Mon", "lat": 11.5380, "lng": 104.9165, "budget": "mid",  "budget_min": 5,  "budget_max": 10, "taste": "savory", "main": "beef",     "cuisine": "khmer",         "allergens": ["soy"],                   "dietary": [],                     "emoji": "🍖", "desc": "Self-grilled marinated meats over charcoal",                  "price": "$5–$10"},
    {"name": "Cha Kdav",               "place": "Night Market Sisowath",    "area": "Daun Penh",   "lat": 11.5700, "lng": 104.9285, "budget": "low",  "budget_min": 2,  "budget_max": 4,  "taste": "spicy",  "main": "stir-fry", "cuisine": "khmer",         "allergens": ["soy", "fish"],          "dietary": [],                     "emoji": "🌶️","desc": "Spicy Khmer stir-fry with chili, basil & protein",           "price": "$2–$4"},
    {"name": "Samlor Machu Kroeung",  "place": "Malis Restaurant",         "area": "BKK1",        "lat": 11.5640, "lng": 104.9282, "budget": "mid",  "budget_min": 5,  "budget_max": 8,  "taste": "sour",   "main": "soup",     "cuisine": "khmer",         "allergens": ["fish", "shellfish"],    "dietary": [],                     "emoji": "🍜", "desc": "Khmer sour soup with lemongrass paste and fresh fish",         "price": "$5–$8"},
    # ── CHINESE ──────────────────────────────────────────────────────
    {"name": "Dim Sum Palace",         "place": "Kampuchea Krom Blvd",       "area": "Chamkar Mon", "lat": 11.5500, "lng": 104.9200, "budget": "mid",  "budget_min": 5,  "budget_max": 15, "taste": "savory", "main": "noodles",  "cuisine": "chinese",       "allergens": ["soy", "gluten"],        "dietary": [],                     "emoji": "🥟", "desc": "Classic dim sum including har gow and siu mai",               "price": "$5–$15"},
    {"name": "Happy Lamb Hotpot",      "place": "Toul Kork",                "area": "Toul Kork",   "lat": 11.5809, "lng": 104.8966, "budget": "mid",  "budget_min": 8,  "budget_max": 22, "taste": "spicy",  "main": "soup",     "cuisine": "chinese",       "allergens": ["soy", "shellfish"],    "dietary": [],                     "emoji": "🍲", "desc": "Chinese lamb hot pot with spicy and mild broth options",       "price": "$8–$22"},
    {"name": "Peking Garden",          "place": "Daun Penh",               "area": "Daun Penh",   "lat": 11.5640, "lng": 104.9290, "budget": "mid",  "budget_min": 6,  "budget_max": 18, "taste": "savory", "main": "beef",     "cuisine": "chinese",       "allergens": ["soy", "gluten"],        "dietary": [],                     "emoji": "🦆", "desc": "Roast duck and classic Chinese BBQ meats",                    "price": "$6–$18"},
    # ── VIETNAMESE ───────────────────────────────────────────────────
    {"name": "Pho Vietnam",            "place": "Street 136, Daun Penh",    "area": "Daun Penh",   "lat": 11.5620, "lng": 104.9280, "budget": "low",  "budget_min": 3,  "budget_max": 7,  "taste": "savory", "main": "noodles",  "cuisine": "vietnamese",   "allergens": ["fish"],                  "dietary": [],                     "emoji": "🍜", "desc": "Authentic Vietnamese pho with beef bone broth",               "price": "$3–$7"},
    {"name": "Banh Mi Kiosk",          "place": "Central Market Area",      "area": "Daun Penh",   "lat": 11.5668, "lng": 104.9254, "budget": "low",  "budget_min": 1,  "budget_max": 3,  "taste": "savory", "main": "sandwich", "cuisine": "vietnamese",   "allergens": ["gluten", "pork"],       "dietary": [],                     "emoji": "🥖", "desc": "Vietnamese baguette sandwich with pâté and pickles",          "price": "$1–$3"},
    # ── JAPANESE ─────────────────────────────────────────────────────
    {"name": "Marugame Udon",          "place": "Toul Kork",                "area": "Toul Kork",   "lat": 11.5848, "lng": 104.9006, "budget": "low",  "budget_min": 3,  "budget_max": 8,  "taste": "savory", "main": "noodles",  "cuisine": "japanese",     "allergens": ["gluten"],               "dietary": [],                     "emoji": "🍜", "desc": "Japanese udon chain with fresh hand-pulled noodles",           "price": "$3–$8"},
    {"name": "Origami Japanese",       "place": "BKK1",                     "area": "BKK1",        "lat": 11.5541, "lng": 104.9268, "budget": "mid",  "budget_min": 8,  "budget_max": 22, "taste": "mild",   "main": "rice",     "cuisine": "japanese",     "allergens": ["fish", "soy"],          "dietary": [],                     "emoji": "🍱", "desc": "Japanese donburi, sushi and teriyaki sets",                    "price": "$8–$22"},
    # ── WESTERN / INTERNATIONAL ──────────────────────────────────────
    {"name": "Burger King Aeon",       "place": "Aeon Mall 1, Chamkar Mon", "area": "Chamkar Mon", "lat": 11.5480, "lng": 104.9254, "budget": "low",  "budget_min": 4,  "budget_max": 10, "taste": "savory", "main": "beef",     "cuisine": "western",      "allergens": ["gluten", "dairy"],      "dietary": [],                     "emoji": "🍔", "desc": "Classic fast food burgers and sides",                          "price": "$4–$10"},
    {"name": "The Pizza Company",      "place": "BKK1",                     "area": "BKK1",        "lat": 11.5543, "lng": 104.9265, "budget": "mid",  "budget_min": 6,  "budget_max": 18, "taste": "savory", "main": "snack",    "cuisine": "western",      "allergens": ["gluten", "dairy"],      "dietary": [],                     "emoji": "🍕", "desc": "Pizza chain with classic and local-inspired toppings",          "price": "$6–$18"},
    {"name": "Green Fork",             "place": "BKK1",                     "area": "BKK1",        "lat": 11.5534, "lng": 104.9268, "budget": "low",  "budget_min": 4,  "budget_max": 12, "taste": "mild",   "main": "snack",    "cuisine": "western",      "allergens": [],                        "dietary": ["vegan","vegetarian"], "emoji": "🌱", "desc": "Vegan cafe with wholesome bowls and wraps",                    "price": "$4–$12"},
    {"name": "Raffles Le Royal",       "place": "Daun Penh",               "area": "Daun Penh",   "lat": 11.5670, "lng": 104.9280, "budget": "high", "budget_min": 15, "budget_max": 50, "taste": "savory", "main": "beef",     "cuisine": "international", "allergens": ["dairy", "gluten"],      "dietary": [],                     "emoji": "🍽", "desc": "Elegant dining at the iconic Raffles Le Royal Hotel",           "price": "$15–$50"},
    {"name": "Haven Restaurant",       "place": "BKK1",                     "area": "BKK1",        "lat": 11.5528, "lng": 104.9264, "budget": "mid",  "budget_min": 5,  "budget_max": 15, "taste": "mild",   "main": "snack",    "cuisine": "international", "allergens": [],                        "dietary": ["vegetarian"],         "emoji": "🍽", "desc": "Training restaurant with a rotating international menu",        "price": "$5–$15"},
    # ── FRENCH ───────────────────────────────────────────────────────
    {"name": "Entrecote Phnom Penh",   "place": "BKK1",                     "area": "BKK1",        "lat": 11.5544, "lng": 104.9271, "budget": "high", "budget_min": 12, "budget_max": 30, "taste": "savory", "main": "beef",     "cuisine": "french",       "allergens": ["dairy"],                "dietary": [],                     "emoji": "🥩", "desc": "Classic French steak-frites restaurant",                       "price": "$12–$30"},
    {"name": "Bonjour Patisserie",     "place": "BKK1",                     "area": "BKK1",        "lat": 11.5540, "lng": 104.9262, "budget": "low",  "budget_min": 3,  "budget_max": 10, "taste": "sweet",  "main": "dessert",  "cuisine": "french",       "allergens": ["gluten","dairy","egg"],"dietary": ["vegetarian"],         "emoji": "🥐", "desc": "French patisserie with croissants and tarts",                   "price": "$3–$10"},
    # ── INDIAN / THAI / KOREAN ───────────────────────────────────────
    {"name": "Taste of India",         "place": "BKK1",                     "area": "BKK1",        "lat": 11.5552, "lng": 104.9258, "budget": "mid",  "budget_min": 5,  "budget_max": 15, "taste": "spicy",  "main": "rice",     "cuisine": "indian",       "allergens": ["dairy", "gluten"],      "dietary": ["vegetarian"],         "emoji": "🍛", "desc": "Butter chicken, biryani and vegetarian Indian classics",        "price": "$5–$15"},
    {"name": "Thai Garden",            "place": "Toul Kork",                "area": "Toul Kork",   "lat": 11.5810, "lng": 104.9010, "budget": "mid",  "budget_min": 6,  "budget_max": 16, "taste": "spicy",  "main": "rice",     "cuisine": "thai",         "allergens": ["fish", "shellfish"],    "dietary": [],                     "emoji": "🌶️","desc": "Tom yum, pad thai and green curry",                           "price": "$6–$16"},
    {"name": "K-BBQ House",            "place": "Chamkar Mon",              "area": "Chamkar Mon", "lat": 11.5490, "lng": 104.9250, "budget": "mid",  "budget_min": 10, "budget_max": 25, "taste": "savory", "main": "beef",     "cuisine": "korean",       "allergens": ["soy", "gluten"],        "dietary": [],                     "emoji": "🥩", "desc": "Korean BBQ with unlimited banchan side dishes",                 "price": "$10–$25"},
    # ── DESSERT / DRINKS ─────────────────────────────────────────────
    {"name": "Mango Tango",            "place": "BKK1",                     "area": "BKK1",        "lat": 11.5540, "lng": 104.9250, "budget": "low",  "budget_min": 2,  "budget_max": 8,  "taste": "sweet",  "main": "dessert",  "cuisine": "western",      "allergens": ["dairy"],                "dietary": ["vegetarian"],         "emoji": "🥭", "desc": "Mango desserts, smoothies and shakes",                          "price": "$2–$8"},
    {"name": "Brown Coffee",           "place": "BKK1",                     "area": "BKK1",        "lat": 11.5545, "lng": 104.9262, "budget": "low",  "budget_min": 3,  "budget_max": 10, "taste": "sweet",  "main": "drink",    "cuisine": "western",      "allergens": ["dairy"],                "dietary": ["vegetarian"],         "emoji": "☕", "desc": "Specialty coffee chain with cakes and pastries",                "price": "$3–$10"},
    {"name": "Durian Palace",          "place": "Chamkar Mon",              "area": "Chamkar Mon", "lat": 11.5490, "lng": 104.9240, "budget": "low",  "budget_min": 2,  "budget_max": 8,  "taste": "sweet",  "main": "dessert",  "cuisine": "khmer",         "allergens": [],                        "dietary": ["vegan","vegetarian"], "emoji": "🍈", "desc": "Durian and tropical fruit desserts",                            "price": "$2–$8"},
]

print(f'✅ Dataset loaded: {len(FOOD_DATASET)} food items')
print(f'   Cuisines available: {len(set(f["cuisine"] for f in FOOD_DATASET))}')
print(f'   Budget tiers: low / mid / high')

# Preview first entry
print('\n📋 Sample entry:')
for k, v in FOOD_DATASET[0].items():
    print(f'   {k:15s}: {v}')

---
## 3. Dataset Exploration
Before building the recommender, let's understand the shape and distribution of our data.

In [ ]:
from collections import Counter

# ── Cuisine distribution ──
cuisine_counts = Counter(f['cuisine'] for f in FOOD_DATASET)
budget_counts  = Counter(f['budget']  for f in FOOD_DATASET)
taste_counts   = Counter(f['taste']   for f in FOOD_DATASET)

print('=== Cuisine Distribution ===')
for c, n in cuisine_counts.most_common():
    bar = '█' * n
    print(f'  {c:15s} {bar} ({n})')

print('\n=== Budget Distribution ===')
for b in ['low', 'mid', 'high']:
    n = budget_counts[b]
    bar = '█' * n
    print(f'  {b:6s} {bar} ({n})')

print('\n=== Taste Distribution ===')
for t, n in taste_counts.most_common():
    bar = '█' * n
    print(f'  {t:8s} {bar} ({n})')

In [ ]:
# ── Visualise dataset distribution ──────────────────────────────────────
ACCENT = '#e83420'
NAVY   = '#1a2b4a'
BG     = '#fff7e7'

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor=BG)
fig.suptitle('NHAM EY KOR BAN — Dataset Overview', fontsize=14,
             fontweight='bold', color=ACCENT, y=1.02)

# Cuisine bar
ax = axes[0]
ax.set_facecolor(BG)
cuisines = [c for c, _ in cuisine_counts.most_common()]
counts   = [n for _, n in cuisine_counts.most_common()]
colors   = [ACCENT if c == 'khmer' else '#4a8abf' for c in cuisines]
ax.barh(cuisines, counts, color=colors, edgecolor='white')
ax.set_title('By Cuisine', fontweight='bold', color=NAVY)
ax.set_xlabel('# items')
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor(BG)

# Budget pie
ax = axes[1]
ax.set_facecolor(BG)
blabels = ['Low ($1–$5)', 'Mid ($5–$15)', 'High ($15+)']
bvals   = [budget_counts['low'], budget_counts['mid'], budget_counts['high']]
bcolors = [ACCENT, '#f39c12', NAVY]
ax.pie(bvals, labels=blabels, colors=bcolors, autopct='%1.0f%%',
       textprops={'fontsize': 9}, startangle=90)
ax.set_title('By Budget', fontweight='bold', color=NAVY)

# Taste bar
ax = axes[2]
ax.set_facecolor(BG)
tastes = [t for t, _ in taste_counts.most_common()]
tvals  = [n for _, n in taste_counts.most_common()]
tcolor = {'savory': ACCENT, 'spicy': '#e67e22', 'sweet': '#27ae60',
          'mild': '#2980b9', 'sour': '#8e44ad', 'stir-fry': '#c0392b'}
tcols  = [tcolor.get(t, '#aaa') for t in tastes]
ax.bar(tastes, tvals, color=tcols, edgecolor='white')
ax.set_title('By Taste', fontweight='bold', color=NAVY)
ax.set_ylabel('# items')
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor(BG)

plt.tight_layout()
plt.savefig('dataset_overview.png', bbox_inches='tight', dpi=120, facecolor=BG)
plt.show()
print('Chart saved as dataset_overview.png')

---
## 4. Scoring Algorithm

### 4.1 Scoring Weights

| Category | Weight | Rationale |
|----------|--------|-----------|
| Budget | 30 pts | Highest priority — affordability is non-negotiable |
| Taste | 25 pts | Strong personal preference |
| Main Course | 25 pts | Core craving (noodles vs rice, etc.) |
| Cuisine | 20 pts | More flexible — people explore cuisines |
| **Total** | **100 pts** | Easy percentage interpretation |

### 4.2 Budget Partial Credit
Budget uses adjacent-level partial scoring to avoid harsh cutoffs:
- Same tier → 30 pts (full match)
- Adjacent tier → 12 pts (40% credit)
- Two levels away → 0 pts

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  SCORING CONSTANTS
# ══════════════════════════════════════════════════════════════════
SCORE_BUDGET  = 30
SCORE_TASTE   = 25
SCORE_MAIN    = 25
SCORE_CUISINE = 20
MAX_SCORE     = SCORE_BUDGET + SCORE_TASTE + SCORE_MAIN + SCORE_CUISINE  # 100

BUDGET_LEVELS = ['low', 'mid', 'high']


# ══════════════════════════════════════════════════════════════════
#  SCORING FUNCTION
# ══════════════════════════════════════════════════════════════════
def score_food(food: dict, budget: str, taste: str, main: str, cuisine: str) -> int:
    """
    Score a single food item against user preferences.
    Returns an integer 0–100.
    
    Algorithm:
      - Budget:  full match = 30, adjacent = 12, no match = 0
      - Taste:   full match or 'any' = 25, else 0
      - Main:    full match or 'any' = 25, else 0
      - Cuisine: full match or 'any' = 20, else 0
    """
    score = 0

    # ── Budget: partial credit for adjacent tier ──────────────────
    u_idx = BUDGET_LEVELS.index(budget)        # user's budget index
    f_idx = BUDGET_LEVELS.index(food['budget'])# food's budget index
    diff  = abs(u_idx - f_idx)                 # tier distance
    if diff == 0:
        score += SCORE_BUDGET                  # 30 pts — exact match
    elif diff == 1:
        score += int(SCORE_BUDGET * 0.4)       # 12 pts — adjacent tier
    # else: 0 pts

    # ── Taste ────────────────────────────────────────────────────
    if taste == 'any' or food['taste'] == taste:
        score += SCORE_TASTE

    # ── Main course ──────────────────────────────────────────────
    if main == 'any' or food['main'] == main:
        score += SCORE_MAIN

    # ── Cuisine ──────────────────────────────────────────────────
    if cuisine == 'any' or food['cuisine'] == cuisine:
        score += SCORE_CUISINE

    return score


# Quick test
test_food = FOOD_DATASET[0]  # Nom Banh Chok
s = score_food(test_food, budget='low', taste='savory', main='noodles', cuisine='khmer')
print(f'Score for "{test_food["name"]}" (perfect match): {s}/{MAX_SCORE} → {s}%')

s2 = score_food(test_food, budget='mid', taste='any', main='any', cuisine='any')
print(f'Score for "{test_food["name"]}" (mid budget, all any): {s2}/{MAX_SCORE} → {s2}%')

---
## 5. Filter + Score + Sort Pipeline

The main pipeline runs in three stages:
1. **Filter** — remove allergen-containing items → O(n·k)
2. **Score** — compute 0–100 score per item → O(n)
3. **Sort** — rank by score descending → O(n log n) via Timsort

In [ ]:
def filter_and_score(budget: str, taste: str, main: str,
                     cuisine: str, allergies: list) -> list:
    """
    Full recommendation pipeline.
    
    Parameters
    ----------
    budget   : 'low' | 'mid' | 'high'
    taste    : 'any' | 'savory' | 'spicy' | 'sweet' | 'sour' | 'mild'
    main     : 'any' | 'rice' | 'noodles' | 'soup' | 'beef' | 'seafood' | ...
    cuisine  : 'any' | 'khmer' | 'chinese' | 'western' | ...
    allergies: list of allergen strings to exclude, e.g. ['fish', 'dairy']
    
    Returns
    -------
    list of food dicts with added 'score' and 'pct' fields, sorted desc.
    """
    results = []

    for food in FOOD_DATASET:
        # ── Stage 1: Allergy filter  O(n·k) ──────────────────────
        if any(a in food['allergens'] for a in allergies):
            continue  # skip unsafe items

        # ── Stage 2: Score  O(n) ─────────────────────────────────
        s = score_food(food, budget, taste, main, cuisine)
        results.append({
            **food,
            'score': s,
            'pct':   round(s / MAX_SCORE * 100)
        })

    # ── Stage 3: Sort  O(n log n) — Python Timsort ───────────────
    results.sort(key=lambda x: x['score'], reverse=True)

    return results


print('✅ filter_and_score() defined')
print(f'   Scoring weights: Budget={SCORE_BUDGET}, Taste={SCORE_TASTE},',
      f'Main={SCORE_MAIN}, Cuisine={SCORE_CUISINE}, Total={MAX_SCORE}')

---
## 6. Running a Sample Query
Let's run the recommender with a concrete set of user preferences.

In [ ]:
# ── Sample user preferences ──────────────────────────────────────────
USER_BUDGET   = 'low'
USER_TASTE    = 'savory'
USER_MAIN     = 'noodles'
USER_CUISINE  = 'khmer'
USER_ALLERGIES = []   # no allergies

results = filter_and_score(
    budget=USER_BUDGET,
    taste=USER_TASTE,
    main=USER_MAIN,
    cuisine=USER_CUISINE,
    allergies=USER_ALLERGIES
)

print(f'Query: budget={USER_BUDGET}, taste={USER_TASTE},',
      f'main={USER_MAIN}, cuisine={USER_CUISINE}')
print(f'Results: {len(results)} items (after allergy filter)\n')

MEDALS = {1: '🥇', 2: '🥈', 3: '🥉'}
print(f'{"Rank":<5} {"Name":<24} {"Score":>5} {"Pct":>5}  {"Budget":<6} {"Taste":<8} {"Main":<10} {"Cuisine"}')
print('-' * 85)
for i, f in enumerate(results[:10], 1):
    medal = MEDALS.get(i, f'#{i} ')
    print(f'{medal:<5} {f["name"]:<24} {f["score"]:>5}/{MAX_SCORE}  {f["pct"]:>3}%  '
          f'{f["budget"]:<6} {f["taste"]:<8} {f["main"]:<10} {f["cuisine"]}')

---
## 7. Data Visualisation — Stacked Score Bar Chart

This chart shows the top-10 results with their score broken into components.  
The grey segment shows "missed" points — making the scoring fully transparent.

In [ ]:
def plot_score_breakdown(results, budget_sel, taste_sel, main_sel, cuisine_sel,
                         title_suffix=''):
    """Stacked bar chart — score breakdown for top-10 results."""
    top10 = results[:10]
    if not top10:
        print('No results to chart.')
        return

    def _cat_scores(food):
        u_idx = BUDGET_LEVELS.index(budget_sel)
        f_idx = BUDGET_LEVELS.index(food['budget'])
        diff  = abs(u_idx - f_idx)
        b = SCORE_BUDGET if diff == 0 else (int(SCORE_BUDGET * 0.4) if diff == 1 else 0)
        t = SCORE_TASTE   if (taste_sel   == 'any' or food['taste']   == taste_sel)   else 0
        m = SCORE_MAIN    if (main_sel    == 'any' or food['main']    == main_sel)    else 0
        c = SCORE_CUISINE if (cuisine_sel == 'any' or food['cuisine'] == cuisine_sel) else 0
        return b, t, m, c

    names    = [f['emoji'] + ' ' + (f['name'][:11] + '…' if len(f['name']) > 11 else f['name'])
                for f in top10]
    cat_data = [_cat_scores(f) for f in top10]
    b_vals   = [d[0] for d in cat_data]
    t_vals   = [d[1] for d in cat_data]
    m_vals   = [d[2] for d in cat_data]
    c_vals   = [d[3] for d in cat_data]
    missed   = [MAX_SCORE - (b+t+m+c) for b, t, m, c in cat_data]
    x        = np.arange(len(names))

    fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
    ax.set_facecolor(BG)

    ax.bar(x, b_vals,                                              color='#e83420', label=f'Budget  (/{SCORE_BUDGET})',  zorder=3)
    ax.bar(x, t_vals, bottom=b_vals,                              color='#f39c12', label=f'Taste   (/{SCORE_TASTE})',   zorder=3)
    ax.bar(x, m_vals, bottom=np.add(b_vals, t_vals),             color='#27ae60', label=f'Main    (/{SCORE_MAIN})',    zorder=3)
    ax.bar(x, c_vals, bottom=np.add(np.add(b_vals,t_vals),m_vals),
                                                                   color='#2980b9', label=f'Cuisine (/{SCORE_CUISINE})', zorder=3)
    ax.bar(x, missed, bottom=np.add(np.add(np.add(b_vals,t_vals),m_vals),c_vals),
                                                                   color='#ddd5c8', label='Not matched', zorder=3)

    for i, food in enumerate(top10):
        ax.text(i, food['score'] + 0.8, str(food['score']),
                ha='center', va='bottom', fontsize=8, fontweight='bold', color='#050505')

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Score breakdown / 100')
    ax.set_title(f'Top 10 — Score Breakdown by Category  {title_suffix}',
                 fontweight='bold', color='#e83420', fontsize=11)
    ax.set_ylim(0, MAX_SCORE + 14)
    ax.yaxis.grid(True, linestyle='--', alpha=0.35, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(fontsize=8, loc='upper right', framealpha=0.85)
    plt.tight_layout()
    plt.savefig('score_breakdown.png', bbox_inches='tight', dpi=120, facecolor=BG)
    plt.show()
    print('Chart saved as score_breakdown.png')


plot_score_breakdown(results, USER_BUDGET, USER_TASTE, USER_MAIN, USER_CUISINE,
                     title_suffix=f'| budget={USER_BUDGET}, taste={USER_TASTE}')

---
## 8. Testing with Different Scenarios

In [ ]:
# ── Scenario A: Vegan, low budget, no allergies ──────────────────────
print('=== SCENARIO A: Vegan-friendly, low budget, any taste ===')
r_a = filter_and_score('low', 'any', 'any', 'any', ['fish', 'egg', 'dairy', 'shellfish', 'pork'])
for i, f in enumerate(r_a[:5], 1):
    print(f'  {i}. {f["emoji"]} {f["name"]:24s} {f["pct"]:3d}%  {f["price"]}')

print()

# ── Scenario B: High budget, savory, beef, western ───────────────────
print('=== SCENARIO B: High budget, savory, beef, western/french ===')
r_b = filter_and_score('high', 'savory', 'beef', 'any', [])
for i, f in enumerate(r_b[:5], 1):
    print(f'  {i}. {f["emoji"]} {f["name"]:24s} {f["pct"]:3d}%  {f["price"]}')

print()

# ── Scenario C: Gluten-free, mid budget, spicy ───────────────────────
print('=== SCENARIO C: Gluten-free, mid budget, spicy ===')
r_c = filter_and_score('mid', 'spicy', 'any', 'any', ['gluten'])
for i, f in enumerate(r_c[:5], 1):
    print(f'  {i}. {f["emoji"]} {f["name"]:24s} {f["pct"]:3d}%  {f["price"]}')

---
## 9. Complexity Analysis

| Stage | Operation | Complexity | Notes |
|-------|-----------|-----------|-------|
| Filter | `any(a in allergens for a in user_allergies)` | O(n·k) | n = items, k = allergens |
| Score | `score_food()` per item | O(n) | Constant-time comparisons |
| Sort | `list.sort()` — Timsort | O(n log n) | Stable, adaptive |
| **Overall** | | **O(n log n)** | Dominated by sort |

With n ≈ 100 items and k ≤ 8 allergens, the system runs in microseconds — well within interactive speed.

In [ ]:
import time

runs = 1000
start = time.perf_counter()
for _ in range(runs):
    filter_and_score('mid', 'savory', 'any', 'khmer', ['gluten', 'dairy'])
elapsed = time.perf_counter() - start

print(f'✅ {runs} full pipeline runs completed')
print(f'   Total time : {elapsed*1000:.2f} ms')
print(f'   Per query  : {elapsed/runs*1000:.4f} ms')
print(f'   Dataset size: {len(FOOD_DATASET)} items')

---
## 10. How to Run the Full GUI Application

```bash
# 1. Install dependencies
pip install matplotlib numpy pillow tkintermapview

# 2. Ensure these files are in the same directory:
#    food_recommendation.py
#    dataSet.py
#    NHAM EY KOR BAN.png

# 3. Launch
python food_recommendation.py
```

---
*NHAM EY KOR BAN — Final Project *